# 06 — Model Comparison & Winner Selection (Validation ONLY)
### AI Interview System — 100% Project-Owned ML Pipeline
This notebook selects the best candidate architecture:
1. Loads validation evaluation results from `reports/candidate_training_report.json`.
2. Loads configurable multi-criteria selection weights from `configs/selection_weights.json`.
3. Normalizes metrics with proper inversion for lower-is-better metrics (Loss, Perplexity, Latency, VRAM).
4. Computes composite weighted scores and determines the single winning own model.
5. Exports decision records: `reports/model_comparison.json` and `reports/best_model_selection.json`.


In [ ]:
# Cell 1: Load Reports & Normalize Metrics
import os
import json
from pathlib import Path
from ml_pipeline_utils import normalize_metrics

WORKSPACE_DIR = Path(os.getcwd())
CAND_REPORT_FILE = WORKSPACE_DIR / "reports" / "candidate_training_report.json"
WEIGHTS_FILE = WORKSPACE_DIR / "configs" / "selection_weights.json"

with open(CAND_REPORT_FILE, "r", encoding="utf-8") as f:
    candidates = json.load(f)["candidates"]

with open(WEIGHTS_FILE, "r", encoding="utf-8") as f:
    weights_config = json.load(f)

# Normalize metrics with lower-is-better inversion
scored_candidates = normalize_metrics(candidates, weights_config)

print("=== CANDIDATE RANKING TABLE (VALIDATION ONLY) ===")
for c in scored_candidates:
    print(f"Rank {c['rank']}: {c['candidate_id']} | Final Score: {c['final_score']:.4f} | Perplexity: {c['raw_metrics']['val_perplexity']}")


In [ ]:
# Cell 2: Select Winning Architecture & Export
winner = scored_candidates[0]
print(f"\nWINNING CANDIDATE SELECTED: {winner['candidate_id']}")

REPORTS_DIR = WORKSPACE_DIR / "reports"

selection_record = {
    "selected_candidate": winner["candidate_id"],
    "model_type": winner["model_type"],
    "parameter_count": winner["parameter_count"],
    "validation_score": winner["final_score"],
    "raw_metrics": winner["raw_metrics"],
    "normalized_metrics": winner["normalized_metrics"],
    "checkpoint_path": f"checkpoints/{winner['candidate_id']}"
}

with open(REPORTS_DIR / "best_model_selection.json", "w", encoding="utf-8") as f:
    json.dump(selection_record, f, indent=2)

with open(REPORTS_DIR / "model_comparison.json", "w", encoding="utf-8") as f:
    json.dump({"ranked_candidates": scored_candidates}, f, indent=2)

print("Exported selection records to reports/best_model_selection.json")
print("Stage 06 Completed Successfully.")
